# Avaliação do sistema de busca com reranker (Colab + GPU)

Roda `recuperacao.sistema_de_busca.avaliacao.avaliar` com o reranker (`BAAI/bge-reranker-v2-m3`), que precisa de GPU.

**Antes de começar**
1. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.
2. Na barra lateral, em **🔑 Secrets**, crie `QDRANT_URL` e `QDRANT_API_KEY` (os mesmos valores do seu `.env`) e libere o acesso deste notebook.
   Se o repositório for privado, crie também `GITHUB_TOKEN` (token com permissão de leitura).
3. Escolha em **2. Código** como trazer o código: clonar do GitHub (precisa ter dado *push*) ou um `.zip` no Drive (para testar mudanças locais ainda não commitadas).

O cache das notas do reranker e as saídas ficam no Google Drive, então uma nova sessão reaproveita o que já foi calculado.

## 1. GPU e Google Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os
PASTA_DRIVE = '/content/drive/MyDrive/tcc_busca'   # cache do reranker + resultados ficam aqui
os.makedirs(f'{PASTA_DRIVE}/resultados', exist_ok=True)

## 2. Código

- **github**: clona a branch indicada (as mudanças precisam estar com *push*).
- **zip**: usa um `.zip` do repositório que você colocou no Drive. No seu PC, compacte a pasta `recuperacao` (pode ignorar `output/` e `__pycache__/`) e envie para `MyDrive/tcc_busca/recuperacao.zip`.

In [ ]:
ORIGEM = 'github'  #@param ["github", "zip"]
BRANCH = 'feat/sistema-de-busca'  #@param {type:"string"}
REPO = 'github.com/gavgms12/tcc_repository.git'  #@param {type:"string"}
ZIP_DRIVE = f'{PASTA_DRIVE}/recuperacao.zip'

import shutil
PROJETO = '/content/tcc_repository'
shutil.rmtree(PROJETO, ignore_errors=True)
if ORIGEM == 'github':
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
        url = f'https://{token}@{REPO}'
    except Exception:
        url = f'https://{REPO}'          # repositório público
    !git clone --depth 1 --branch {BRANCH} {url} {PROJETO}
else:
    os.makedirs(PROJETO)
    !unzip -q "{ZIP_DRIVE}" -d {PROJETO}
%cd {PROJETO}
!ls recuperacao/sistema_de_busca

## 3. Dependências
O Colab já vem com `torch` com CUDA; aqui só entra o resto.

In [ ]:
!pip install -q -r requirements-busca.txt pandas scipy python-dotenv

## 4. Variáveis de ambiente

In [ ]:
from google.colab import userdata
os.environ['QDRANT_URL'] = userdata.get('QDRANT_URL')
os.environ['QDRANT_API_KEY'] = userdata.get('QDRANT_API_KEY')
# cache das notas do reranker no Drive (sobrevive entre sessões; ver config.RERANK_CACHE_PATH)
os.environ['RERANK_CACHE_PATH'] = f'{PASTA_DRIVE}/rerank_scores.sqlite'
os.environ['PYTHONPATH'] = PROJETO

from qdrant_client import QdrantClient
c = QdrantClient(url=os.environ['QDRANT_URL'], api_key=os.environ['QDRANT_API_KEY'])
print('Qdrant OK:', len(c.get_collections().collections), 'coleções')

## 4b. Consultas manuais (só para `CONJUNTO = manual`)

Escolha de onde vem o `consultas_manuais.json`:
- **upload**: abre o seletor de arquivos; escolha o JSON do seu PC (`recuperacao/sistema_de_busca/avaliacao/consultas_manuais.json`). Ele também é salvo no Drive para as próximas sessões.
- **drive**: usa `MyDrive/tcc_busca/consultas_manuais.json` (o último enviado).
- **repositorio**: usa o arquivo que veio com o código (clone/zip).

In [ ]:
FONTE_MANUAIS = 'upload'  #@param ["upload", "drive", "repositorio"]
MANUAIS_DRIVE = f'{PASTA_DRIVE}/consultas_manuais.json'
ARQUIVO_MANUAIS = 'recuperacao/sistema_de_busca/avaliacao/consultas_manuais.json'

if FONTE_MANUAIS == 'upload':
    from google.colab import files
    enviado = files.upload()                      # escolha o consultas_manuais.json
    nome = next(iter(enviado))
    with open(MANUAIS_DRIVE, 'wb') as f:
        f.write(enviado[nome])
    ARQUIVO_MANUAIS = MANUAIS_DRIVE
elif FONTE_MANUAIS == 'drive':
    ARQUIVO_MANUAIS = MANUAIS_DRIVE

import json
with open(ARQUIVO_MANUAIS, encoding='utf-8') as f:
    consultas = json.load(f).get('consultas', [])
validas = [c for c in consultas if (c.get('texto') or c.get('temas')) and (c.get('alta') or c.get('media'))]
print(f'{ARQUIVO_MANUAIS}: {len(consultas)} consultas, {len(validas)} com texto/temas e esperados')
for c in validas:
    print(' -', (c.get('texto') or ', '.join(c['temas']))[:90])

## 5. Rodar a avaliação

| Parâmetro | O que faz |
|---|---|
| `CONJUNTO` | `tcc`, `ic`, `manual` ou `ambos` (= tcc + ic); `manual` usa o JSON escolhido na 4b |
| `N` | quantos TCCs sortear (0 = todos os 373); semente fixa, então as mesmas consultas a cada rodada |
| `MODOS` | modos de busca, separados por espaço |
| `SO_PRE` | marque para rodar sem reranker (comparação rápida) |
| `EXTRA` | qualquer outro argumento do `avaliar`, ex.: `--uniao media decaimento`, `--n-ic 50`, `--k 10` |

Cada modo roda sem e com rerank (`hibrida` e `hibrida+rerank`) e o Wilcoxon compara tudo com `hibrida+rerank`.
A 1ª rodada é a mais lenta (baixa os modelos e preenche o cache); as seguintes reaproveitam as notas do cache.

No conjunto `manual` o log também traz o ranking de cada consulta, com ★/☆ nos esperados.

In [ ]:
CONJUNTO = 'manual'  #@param ["tcc", "ic", "manual", "ambos"]
N = 150  #@param {type:"integer"}
MODOS = 'densa esparsa hibrida'  #@param {type:"string"}
SO_PRE = False  #@param {type:"boolean"}
EXTRA = ''  #@param {type:"string"}

arquivo = f'--arquivo "{ARQUIVO_MANUAIS}"' if 'ARQUIVO_MANUAIS' in globals() else ''
args = f'--conjunto {CONJUNTO} --n {N} {arquivo} --modos {MODOS} {"--so-pre" if SO_PRE else ""} {EXTRA}'
print('python -m recuperacao.sistema_de_busca.avaliacao.avaliar', args)
!python -m recuperacao.sistema_de_busca.avaliacao.avaliar {args}

## 6. Salvar os resultados no Drive (e baixar)

In [ ]:
import glob
pastas = sorted(glob.glob('recuperacao/output/avaliacao_*'))
ultima = pastas[-1]
destino = f'{PASTA_DRIVE}/resultados/{os.path.basename(ultima)}'
shutil.copytree(ultima, destino, dirs_exist_ok=True)
print('Copiado para', destino)

arquivo_zip = shutil.make_archive(f'/content/{os.path.basename(ultima)}', 'zip', ultima)
from google.colab import files
files.download(arquivo_zip)

## 7. (Opcional) Ver o resumo aqui

In [ ]:
import pandas as pd
pd.read_csv(f'{ultima}/resumo.csv')